# Evaluating COMET score on generated translations

In [ ]:
!pip -q uninstall -y numpy unbabel-comet comet
!pip -q install --no-cache-dir "numpy==1.26.4"
!pip -q install --no-cache-dir "protobuf>=4.24.4,<5" "unbabel-comet==2.2.7"
import os; os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 173.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.

In [1]:
import numpy as np, google.protobuf
print("numpy:", np.__version__)
print("protobuf:", google.protobuf.__version__)

numpy: 1.26.4
protobuf: 4.25.8


In [2]:
from comet import download_model, load_from_checkpoint
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
import pandas as pd
PRED_PATH = "/content/drive/MyDrive/ds266_idiom_mt/qual_preds/preds_5models.csv"
pred_df = pd.read_csv(PRED_PATH)
pred_df.head()

,example_id,group,source,reference,pred_baseline,pred_idiom_only_v1,pred_lora_r4_stage2,pred_lora_r8_stage2,pred_lora_r16_stage2,disagree_len
0,0,idioms_test,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...","Papa sagt, ich bin nicht für das Militär gesch...","Dad sagt, ich bin nicht für das Militär geeign...","Dad sagt, ich bin nicht für das Militär geeign...","Dad sagt, ich bin nicht für das Militär geeign...",1.200000
1,1,idioms_test,She's always trying to pass the buck and I'm s...,"Sie versucht immer, den schwarzen Peter weiter...","Sie versucht immer, das Geld zu verteilen und ...","Sie versucht immer, den Dollar zu geben, und i...","Sie versucht immer, das Geld zu geben, und ich...","Sie versucht immer, den Dollar zu überreichen,...","Sie versucht immer, den Dollar zu überreichen,...",3.249615
2,2,idioms_test,He has a reputation as a loose cannon whose co...,"Er gilt als tickende Zeitbombe, dessen Äußerun...","Er hat einen Ruf als lose Kanone, deren Kommen...","Er hat den Ruf, eine lockere Kanone zu sein, d...","Er hat einen Ruf als lockere Kanone, deren Kom...","Er hat einen Ruf als lose Kanone, deren Kommen...","Er hat einen Ruf als lockere Kanone, deren Kom...",5.114685
3,3,idioms_test,He's been working late with her every night th...,Er hat diese Woche jeden Abend mit ihr lange g...,Er hat diese Woche jede Nacht mit ihr gearbeit...,Er hat diese Woche jede Nacht zu spät mit ihr ...,Ich rieche eine Ratte.,Er hat diese Woche jede Nacht mit ihr zu spät ...,Er hat diese Woche jede Nacht mit ihr zu tun.,23.039097
4,4,idioms_test,Burning the candle at both ends is a recipe fo...,"Raubbau mit der Gesundheit zu treiben, ist ein...",Das Verbrennen der Kerze an beiden Enden ist e...,Das Verbrennen der Kerze an beiden Enden ist e...,Das Verbrennen der Kerze an beiden Enden ist e...,Das Verbrennen der Kerze an beiden Enden ist e...,Das Verbrennen der Kerze an beiden Enden ist e...,0.000000


## Load COMET Model

In [7]:
# Load COMET model
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path).to(device).eval()
print("Loaded:", model_path)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


Loaded: /root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt


## Score Predictions

In [8]:
pred_cols = [c for c in pred_df.columns if c.startswith("pred_")]
pred_cols

['pred_baseline',
 'pred_idiom_only_v1',
 'pred_lora_r4_stage2',
 'pred_lora_r8_stage2',
 'pred_lora_r16_stage2']

In [9]:
def comet_scores_for_column(df, pred_col, batch_size=16):
    data = [{"src": r["source"], "mt": r[pred_col], "ref": r["reference"]} for _, r in df.iterrows()]
    out = comet_model.predict(
        data,
        batch_size=batch_size,
        gpus=1 if device == "cuda" else 0
    )
    return out.scores

rows = []
for c in pred_cols:
    # overall
    s_all = comet_scores_for_column(pred_df, c, batch_size=16)
    rows.append({"model": c.replace("pred_",""), "split": "ALL", "comet_mean": float(np.mean(s_all)), "n": len(s_all)})

    # by group
    for grp in pred_df["group"].unique():
        sub = pred_df[pred_df["group"] == grp]
        s = comet_scores_for_column(sub, c, batch_size=16)
        rows.append({"model": c.replace("pred_",""), "split": grp, "comet_mean": float(np.mean(s)), "n": len(s)})

comet_summary = pd.DataFrame(rows).sort_values(["split","comet_mean"], ascending=[True, False])


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/

,model,split,comet_mean,n
3,idiom_only_v1,ALL,0.807304,50
12,lora_r16_stage2,ALL,0.789763,50
9,lora_r8_stage2,ALL,0.787165,50
0,baseline,ALL,0.782859,50
6,lora_r4_stage2,ALL,0.780073,50
4,idiom_only_v1,idioms_test,0.762717,25
13,lora_r16_stage2,idioms_test,0.732651,25
1,baseline,idioms_test,0.732551,25
10,lora_r8_stage2,idioms_test,0.732034,25
7,lora_r4_stage2,idioms_test,0.727875,25


In [10]:
comet_summary

,model,split,comet_mean,n
3,idiom_only_v1,ALL,0.807304,50
12,lora_r16_stage2,ALL,0.789763,50
9,lora_r8_stage2,ALL,0.787165,50
0,baseline,ALL,0.782859,50
6,lora_r4_stage2,ALL,0.780073,50
4,idiom_only_v1,idioms_test,0.762717,25
13,lora_r16_stage2,idioms_test,0.732651,25
1,baseline,idioms_test,0.732551,25
10,lora_r8_stage2,idioms_test,0.732034,25
7,lora_r4_stage2,idioms_test,0.727875,25


Interpretation:
- On idiom-focused sentences, COMET shows large gains for full idiom-only fine-tuning
- But is comparatively insensitive among LoRA variants and baseline despite their BLEU/chrF differences

## Export COMET results

In [11]:
OUT = "/content/drive/MyDrive/ds266_idiom_mt/qual_preds/COMET_summary.csv"
comet_summary.to_csv(OUT, index=False)
print("Saved:", OUT)

Saved: /content/drive/MyDrive/ds266_idiom_mt/qual_preds/COMET_summary.csv
